### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** You don't just 'use Adam'. You understand that Adam generalizes worse than SGD for some tasks. You know that **Lion** simplifies Adam but requires careful weight decay tuning. You use **SAM** when generalization is more important than training speed. And most importantly, you know that the LR schedule is often as important as the optimizer itself.

**Mental Model:** 
- **Vanillia SGD:** Gravity pulls you down.
- **Momentum:** You carry speed through small bumps.
- **Adam:** You have a GPS that shrinks your steps on steep cliffs and grows them in flat valleys.
- **SAM:** You check the area around you; if moving an inch makes the loss explode, you move elsewhere (seeking 'flat' minima).

**Common Junior Pitfall:** Not using gradient clipping when training Recurrent Networks (RNNs) or Transformers. Huge gradients will eventually cause a 'Weight Explosion' (NaN), crashing the training run. Norm-clipping to 1.0 is the standard solution.

---


## 1. The Physics of SGD vs SGDM (Momentum)

Imagine a ball in a ravine. SGD bounces violently between the walls because the gradient is huge in the vertical direction but tiny in the horizontal (the direction we want to go).

**Momentum** adds physical mass:
1. $v_{t} = \beta v_{t-1} + (1 - \beta) \nabla L$ 
2. $W_{t} = W_{t-1} - \alpha v_{t}$

The vertical bounces cancel each other out over time, while the horizontal speed builds up. This is **Temporal Averaging**.

📈 **Production Signal:** While Adam is the default, researchers at Meta and DeepMind often use SGD with Momentum for final production Vision models because it often finds better generalizing 'flat' minima than Adam.


## 📑 Table of Contents

* [🎓 Socratic Deep Check](#-socratic-deep-check)
  * [🎓 Junior to Senior: Concept Bridge](#-junior-to-senior-concept-bridge)
* [1. The Physics of SGD vs SGDM (Momentum)](#1-the-physics-of-sgd-vs-sgdm-momentum)
* [2. Lion — Sign-Based Momentum](#2-lion-sign-based-momentum)
  * [🎓 Socratic Deep Check](#-socratic-deep-check)
* [3. Adaptive Rates (Adam & AdamW)](#3-adaptive-rates-adam-adamw)
  * [The AdamW Correction](#the-adamw-correction)
* [4. Loss Landscape Sharpness & Flat Minima (SAM)](#4-loss-landscape-sharpness-flat-minima-sam)
* [5. Learning Rate Schedules — The Full Taxonomy](#5-learning-rate-schedules-the-full-taxonomy)
* [6. Gradient Clipping — When and How](#6-gradient-clipping-when-and-how)
* [✅ Knowledge Check](#-knowledge-check)
  * [Q1: Why use Warmup?](#q1-why-use-warmup)
  * [Q2: Lion vs Adam Memory](#q2-lion-vs-adam-memory)
  * [Q3: SAM adversarial updates](#q3-sam-adversarial-updates)
* [🔨 Practical Practice](#-practical-practice)
  * [Tier 1: Beginner](#tier-1-beginner)
  * [Tier 2: Intermediate](#tier-2-intermediate)
  * [Tier 3: Advanced](#tier-3-advanced)

---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

def loss_fn(x, y): return 10 * x**2 + 0.1 * y**2
def grad_fn(x, y): return np.array([20 * x, 0.2 * y])

def simulate_opt(name, lr, beta=0.9, steps=30):
    pos = np.array([-8.0, 9.0])
    path = [pos.copy()]
    m = np.zeros(2)
    
    for _ in range(steps):
        g = grad_fn(*pos)
        if name == 'sgd':
            pos = pos - lr * g
        elif name == 'mom':
            m = beta * m + (1 - beta) * g
            pos = pos - lr * m
        elif name == 'lion':
            # Lion Update: W = W - lr * sign(beta * m + (1-beta) * g)
            update = np.sign(beta * m + (1 - beta) * g)
            pos = pos - lr * update
            # Lion momentum: update m for next step
            m = 0.99 * m + (1 - 0.99) * g # beta2 is usually 0.99 in Lion
        path.append(pos.copy())
    return np.array(path)

path_sgd = simulate_opt('sgd', 0.08)
path_mom = simulate_opt('mom', 0.02)
path_lion = simulate_opt('lion', 0.5) # Lion takes equal-sized steps

plt.figure(figsize=(10, 6))
X, Y = np.meshgrid(np.linspace(-10, 10, 100), np.linspace(-10, 10, 100))
plt.contour(X, Y, loss_fn(X, Y), levels=20, cmap='viridis', alpha=0.3)
plt.plot(path_sgd[:,0], path_sgd[:,1], 'ro-', label='SGD', markersize=3)
plt.plot(path_mom[:,0], path_mom[:,1], 'bo-', label='Momentum', markersize=3)
plt.plot(path_lion[:,0], path_lion[:,1], 'go-', label='Lion (Sign-based)', markersize=3)
plt.legend(); plt.title("The Ravine: Lion vs Adam vs SGD")
plt.show()

"""
What just happened?
Notice Lion's green path. Unlike SGD which bounces, Lion takes fixed-magnitude steps
in the 'vibe' of the gradient. In the horizontal direction where gradients are tiny,
Lion still takes full-sized steps, allowing it to cross the ravine much faster than others.
"""

## 2. Lion — Sign-Based Momentum

Discovered via evolutionary search by Google (2023), Lion (Evolved Sign Momentum) is simpler and more memory-efficient than Adam.

**Update Rule:**
1. $u_t = \text{sign}(\beta_1 m_{t-1} + (1-\beta_1) \nabla L)$
2. $W_t = W_{t-1} - \alpha (u_t + \lambda W_{t-1})$
3. $m_t = \beta_2 m_{t-1} + (1-\beta_2) \nabla L$

**Why is this revolutionary?**
- **Memory:** It only stores ONE momentum vector ($m_t$). Adam stores TWO ($m_t$ and $v_t$). This reduces optimizer memory overhead by 50%.
- **Robustness:** Because it uses the `sign()`, every parameter update is exactly $+\alpha$ or $-\alpha$. This prevents issues with sparse gradients where some parameters never move.

📈 **Production Signal:** Google used Lion to train Vision Transformers, reporting up to **10x compute efficiency gains** over Adam in some massive-scale tasks.

### 🎓 Socratic Deep Check
> *"If every parameter update is fixed at precisely +lr or -lr, doesn't the network lose the ability to make 'refined' small updates as it gets closer to 0 loss? How does Lion handle convergence in the final steps compared to Adam's decaying step sizes?"*

## 3. Adaptive Rates (Adam & AdamW)

Adam (Adaptive Moment Estimation) tracks both the **Mean** (Momentum) and **Variance** (Friction) of gradients. It's the undisputed default for modern deep learning.

**Key Insight:** Adam calculates a per-parameter adaptive learning rate. $\text{step} = \frac{m}{\sqrt{v} + \epsilon}$. This effectively flattens the landscape, making ravines look like bowls.

### The AdamW Correction
Decoupling Weight Decay: Standard Adam treats weight decay as part of the gradient, which gets "normalized" away in the denominator. AdamW manually subtracts weight decay *after* the adaptive step, ensuring regularization actually works for Transformers.

📈 **Production Signal:** AdamW is the backbone for training every LLM from GPT-3 to Llama 3.


## 4. Loss Landscape Sharpness & Flat Minima (SAM)

Generalization depends on the "sharpness" of the minimum. A **Flat Minimum** means if your weights shift slightly (e.g. from noise), your loss barely changes. A **Sharp Minimum** means a tiny shift crashes performance.

**SAM (Sharpness-Aware Minimization)** finds flat minima by looking for the worst-case loss in a small neighborhood and minimizing THAT:
1.  **Perturb:** $\epsilon = \rho \frac{\nabla L}{||\nabla L||}$
2.  **Lookahead:** $W_{adv} = W + \epsilon$
3.  **Update:** $W = W - \alpha \nabla L(W_{adv})$

📈 **Production Signal:** SAM is used by state-of-the-art vision teams to push the final 1-2% accuracy on ImageNet competition models where generalization is key.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class SAMOptimizer(optim.Optimizer):
    def __init__(self, params, base_optimizer, rho=0.05, **kwargs):
        defaults = dict(rho=rho, **kwargs)
        super(SAMOptimizer, self).__init__(params, defaults)
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + 1e-12)
            for p in group["params"]:
                if p.grad is None: continue
                e_w = p.grad * scale
                p.add_(e_w)  # climb to worst-case state
                self.state[p]["e_w"] = e_w
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None: continue
                p.sub_(self.state[p]["e_w"])  # restore weights
        self.base_optimizer.step()
        if zero_grad: self.zero_grad()

    def _grad_norm(self):
        shared_device = self.param_groups[0]["params"][0].device
        norm = torch.norm(torch.stack([torch.norm(p.grad).to(shared_device) for group in self.param_groups for p in group["params"] if p.grad is not None]), p=2)
        return norm

"""
What just happened?
We implemented the SAM wrapper. It works in two steps: 
1. 'First Step' moves weights temporarily toward 'worst-case' noise.
2. 'Second Step' computes gradients at that noisy point and applies the actual update.
This forces the optimizer to find regions that are resilient to noise (flat minima).
"""

## 5. Learning Rate Schedules — The Full Taxonomy

Schedules vary the step size over time to achieve fast exploration and stable convergence.

1.  **Cosine Annealing:** Smoothly decays LR toward zero. High stability.
2.  **Warmup-Constant:** Zero to Max LR linearly, then stays flat. Best for early BERT.
3.  **OneCycle:** Peak LR in the middle. "Super Convergence" phenomenon.
4.  **Cyclical (CLR):** Periodic ups and downs. Bypasses saddle points by "resetting" speed.

📈 **Production Signal:** OneCycle is the "secret sauce" behind reaching high accuracy in record-breaking times in Fast.ai and Kaggle competitions.


In [ ]:
def cosine_schedule(t, total_steps, max_lr): 
    return max_lr * 0.5 * (1 + np.cos(np.pi * t / total_steps))

def one_cycle(t, total_steps, max_lr):
    pct = t / total_steps
    if pct < 0.3:
        return (pct / 0.3) * max_lr
    return max_lr * (1 - (pct - 0.3) / 0.7)

steps = np.arange(1000)
plt.figure(figsize=(12, 4))
plt.plot(steps, [cosine_schedule(s, 1000, 0.1) for s in steps], label="Cosine Annealing")
plt.plot(steps, [one_cycle(s, 1000, 0.1) for s in steps], label="OneCycle (Warmup+Decay)")
plt.title("The Modern Schedule Taxonomy"); plt.ylabel("Learning Rate"); plt.legend()
plt.show()

"""
What just happened?
We plotted current industry-standard schedules. Warmup (OneCycle) prevents 'divergence' 
in early training when batch norms are uninitialized and weights are noisy.
"""

## 6. Gradient Clipping — When and How

Deep networks, especially RNNs and LSTMs, can produce massive gradients during backprop through many timesteps. These gradients overwrite weights with huge numbers (Weight Explosion), leading to **NaNs**.

**Two Types:**
1.  **Value Clipping:** Force every grad $g_i \in [-c, c]$. *Warning: This changes the direction of the gradient vector!*
2.  **Norm Clipping:** If $||g|| > c$, rescale the entire vector $g = g \cdot \frac{c}{||g||}$. *Advantage: This preserves the direction (the 'vibe') but caps the magnitude.*

📈 **Production Signal:** LLM training runs (like GPT-4) would crash within minutes without Global Norm Clipping (usually set to 1.0).


In [ ]:
def clip_grad_norm(params, max_norm):
    total_norm = np.linalg.norm([np.linalg.norm(p.grad) for p in params])
    if total_norm > max_norm:
        scale = max_norm / (total_norm + 1e-6)
        for p in params: p.grad *= scale

"""
What just happened?
We implemented norm clipping. Notice how we compute a 'global' norm for the entire layer
and apply a single scaling factor. This prevents 'gradient spikes' from destroying
weeks of work in a single step.
"""

---
## ✅ Knowledge Check

### Q1: Why use Warmup?
<details><summary>👉 Answer</summary>
At start, weights are random and loss landscapes are messy. Large steps early on can push weights into regions so bad the model can never recover ('divergence'). Warmup starts slow while Batch Norm statistics stabilize and the model 'finds its footing'.
</details>

### Q2: Lion vs Adam Memory
<details><summary>👉 Answer</summary>
Adam stores Momentum ($m$) and Variance ($v$). For a 1B parameter model, this is 2B extra floats (8GB). Lion only stores $m$, saving 4GB of VRAM — often the difference between being able to train on a specific GPU or not.
</details>

### Q3: SAM adversarial updates
<details><summary>👉 Answer</summary>
SAM intentionally looks for the 'sharpest' nearby bump (adversarial step) and moves to minimize that specifically. This 'smooths out' the weights, ensuring they land in a broad, reliable valley (flat minimum) rather than a narrow, fragile hole (sharp minimum).
</details>


---
## 🔨 Practical Practice

### Tier 1: Beginner
1. Implement **Sign-SGD** (the simplest form of Lion) and observe its path on the ravine problem.
2. Modify the LR schedule plot to include a **Triangular Cyclical LR**.

### Tier 2: Intermediate
1. **Lion vs Adam Efficiency:** Train a Vision Transformer (ViT) on CIFAR-10. Record the time and memory usage for both. Show that Lion uses less VRAM.
2. **Clipping Crisis:** Deliberately create an unstable RNN task (e.g. counting to 500). Show that Norm Clipping prevents NaNs where Value Clipping fails.

### Tier 3: Advanced
1. **RAdam implementation:** Implement **Rectified Adam (RAdam)** from scratch. Prove why its automated 'warmup' component makes manual warmup unnecessary for many tasks.
2. **Lookahead Optimizer:** Implement the 'Lookahead' wrapper (Michael Zhang et al. 2019). Combine it with SGD and show that it achieves Adam-like stability while maintaining SGD-like generalization.
